In [1]:
import torch
from transformers import AutoConfig
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForMaskedLM
from transformers import AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [4]:
md_id = "sentence-transformers/all-MiniLM-L6-v2"

print("Model ID:", md_id)


Model ID: sentence-transformers/all-MiniLM-L6-v2


In [11]:
dir(md_id)

['__add__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmod__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'capitalize',
 'casefold',
 'center',
 'count',
 'encode',
 'endswith',
 'expandtabs',
 'find',
 'format',
 'format_map',
 'index',
 'isalnum',
 'isalpha',
 'isascii',
 'isdecimal',
 'isdigit',
 'isidentifier',
 'islower',
 'isnumeric',
 'isprintable',
 'isspace',
 'istitle',
 'isupper',
 'join',
 'ljust',
 'lower',
 'lstrip',
 'maketrans',
 'partition',
 'removeprefix',
 'removesuffix',
 'replace',
 'rfind',
 'rindex',
 'rjust',
 'rpartition',
 'rsplit',
 'rstrip',
 'split',
 'splitlines',
 'startswith',
 'stri

In [13]:
config = AutoConfig.from_pretrained(md_id)

print("Model type:", config.model_type)
print("Vocabulary size:", config.vocab_size)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)

Model type: bert
Vocabulary size: 30522
Hidden size: 384
Number of layers: 6


In [20]:
Tokenizer= AutoTokenizer.from_pretrained(md_id)

txt=["the world is peaceful",
"don't we all just love ice cream?"]

tokens = Tokenizer(
    txt,
    padding=True,
    truncation=True,
    return_tensors= "pt"
)


print(tokens)

print("Input IDs:")
print(tokens["input_ids"])

print("\nAttention Mask:")
print(tokens["attention_mask"])

{'input_ids': tensor([[ 101, 1996, 2088, 2003, 9379,  102,    0,    0,    0,    0,    0,    0],
        [ 101, 2123, 1005, 1056, 2057, 2035, 2074, 2293, 3256, 6949, 1029,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Input IDs:
tensor([[ 101, 1996, 2088, 2003, 9379,  102,    0,    0,    0,    0,    0,    0],
        [ 101, 2123, 1005, 1056, 2057, 2035, 2074, 2293, 3256, 6949, 1029,  102]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [22]:
model = AutoModel.from_pretrained(md_id)
model = model.to(device)

#Evaluation Mode
model.eval()

inputs = {key: value.to(device) for key, value in tokens.items()}

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.last_hidden_state

print("Hidden states shape:")
print(hidden_states.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hidden states shape:
torch.Size([2, 12, 384])


### Sentence Similarity

To calculate sentence similarity, we first obtain the sentence embeddings (vector representations) from our `AutoModel` for each sentence. Then, we can use cosine similarity to measure how similar these vectors are.

In [30]:
from torch.nn.functional import cosine_similarity


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output # model_output is outputs.last_hidden_state, shape [batch_size, seq_len, hidden_size]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Get sentence embeddings
sentence_embeddings = mean_pooling(outputs.last_hidden_state, inputs['attention_mask'])

print("Sentence Embeddings Shape:", sentence_embeddings.shape)

# Calculate cosine similarity
similarity_score = cosine_similarity(sentence_embeddings[0].unsqueeze(0), sentence_embeddings[1].unsqueeze(0))

print(f"Sentence 1: '{txt[0]}'")
print(f"Sentence 2: '{txt[1]}'")
print(f"Cosine Similarity: {similarity_score.item():.4f}")

Sentence Embeddings Shape: torch.Size([2, 384])
Sentence 1: 'the world is peaceful'
Sentence 2: 'don't we all just love ice cream?'
Cosine Similarity: 0.1047
